Notebook to examine behavioral results of a given imaging session

last edit WMS 1/18/26

In [ ]:
#imports
import pandas as pd
import numpy as np
import os
import scipy.io
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import scipy.stats as stats
import pickle
import utils

In [ ]:
#import data
bento_path = r"C:\miniscope_files\analyzed\7981\2025_03_25\bento"

datapath = os.path.join(bento_path,"analysis_results")

prep_path = os.path.join(datapath, "data_prep")

In [ ]:
#define dir to save results into
behpath = os.path.join(datapath, "behavioral_results")

In [ ]:
#make the dir
if not os.path.isdir(behpath):
    os.mkdir(behpath)

In [ ]:
#define dictionary to save results into
resdict = {}

In [ ]:
raster_file = os.path.join(prep_path, "annot_rasters_updated.csv")
rdf = pd.read_csv(raster_file,index_col=0)

In [ ]:
rdf

In [ ]:
cam_dir = os.path.join(bento_path, "minicam2\\timeStamps.csv") #"minicam1\\timeStamps.csv"
cam_ts = pd.read_csv(cam_dir)

In [ ]:
cam_ts

In [ ]:
#if cam1_ts.shape[0] == raster_df.shape[1]:
if cam_ts.shape[0] == rdf.shape[1]:
    print("Behavioral data loaded in as expected")
else:
    print("WARNING: Behavioral data timestamps frame length does not reflect the # of frames in the annotated behavioral data. Please double-check timestamps file and behavioral annoptations belong to the same session!")

In [ ]:
cam_fr = utils.approx_fr(cam_ts)
cam_fr

Make separate df for each five minute block of the two social trials

In [ ]:
#get the first and last columns for social1
social1_mask = rdf.loc['social1'] == 1 #mask for social1 trial
social1_df = rdf.loc[:,social1_mask].copy()
social1_colnames = np.where(social1_mask)[0] #get the column indices (not names) for social1
first_s1_frame = social1_colnames[0]
last_s1_frame = social1_colnames[-1]
print(first_s1_frame,last_s1_frame)

In [ ]:
social1_df

In [ ]:
#get the first and last columns for social2
social2_mask = rdf.loc['social2'] == 1 #mask for social1 trial
social2_df = rdf.loc[:,social2_mask].copy()
social2_colnames = np.where(social2_mask)[0] #get the column indices (not names) for social1
first_s2_frame = social2_colnames[0]
last_s2_frame = social2_colnames[-1]
print(first_s2_frame,last_s2_frame)

Plot test animal behavior over the two social trials

In [ ]:
#generate df for plotting and rename the columns to skip the intervening barriers trial
social_frames_df = pd.concat([social1_df,social2_df],axis=1)
social_frames_df.columns = range(len(social_frames_df.columns))
separator_frame = social1_df.shape[1]

In [ ]:
social_frames_df

In [ ]:
fig, ax = utils.plot_behavior_raster(
    social_frames_df,
    behaviors=['partner_active', 'novel_active'],
    fr=cam_fr,
    separator_frame=separator_frame,
    set_title=True, title="Test Animal Behavior During Social Trials",
    save_path=os.path.join(behpath, 'active_interactions_social_trials.png')
)

In [ ]:
fig, ax = utils.plot_behavior_raster(
    social_frames_df,
    behaviors=['partner_active', 'novel_active', 'partner_chamb', 'novel_chamb', 'center'],
    fr=cam_fr,
    separator_frame=separator_frame,
    set_title=True, title="Test Animal Behavior During Social Trials",
    save_path=os.path.join(behpath, 'active_interactions_and_chambers_social_trials.png')
)

In [ ]:
fig, ax = utils.plot_behavior_raster(
    social_frames_df,
    behaviors=['active','left', 'right', 'center'],
    fr=cam_fr,
    separator_frame=separator_frame,
    set_title=True, title="Test Animal Behavior During Social Trials",
    save_path=os.path.join(behpath, 'chambers_social_trials_with_unlabeled_active_interactions.png')
)

In [ ]:
fig, ax = utils.plot_behavior_raster(
    rdf,
    behaviors=['active','left', 'right', 'center'],
    fr=cam_fr,
    #separator_frame=separator_frame,
    set_title=True, title="Test Animal Behavior During All Trials",
    save_path=os.path.join(behpath, 'chambers_social_trials_with_unlabeled_active_interactions_without_trials.png')
)

In [ ]:
fig, ax = utils.plot_behavior_raster(
    rdf,
    behaviors=['active','left', 'right', 'center', 'empty1', 'social1', 'social2', 'empty2'],
    fr=cam_fr,
    #separator_frame=separator_frame,
    set_title=True, title="Test Animal Behavior During All Trials",
    save_path=os.path.join(behpath, 'chambers_social_trials_with_unlabeled_active_interactions_with_trials.png')
)

In [ ]:
#get the frames corresponding to five minutes after the start of both trials (as well as the exact timestamp and the exact offset time from start to 'five minutes')
s1_fivemin_frame, s1_fivemin_ts, s1_firstfive_time = utils.get_offset_frame(cam_ts,start_frame=first_s1_frame,time_offset=300,units='sec')
s2_fivemin_frame, s2_fivemin_ts, s2_firstfive_time = utils.get_offset_frame(cam_ts,start_frame=first_s2_frame,time_offset=300,units='sec')

print(s1_firstfive_time)
print(s2_firstfive_time)

In [ ]:
#sanity check that there is ~5 min (300 seconds) in each chunk

In [ ]:
s1_start_ts = cam_ts.loc[cam_ts['Frame Number']==first_s1_frame, 'Time Stamp (ms)'].values[0]
s1_firstfive_time_calc = (s1_fivemin_ts - s1_start_ts)/1000

print(f'real time for estimated first five minutes (300 seconds) of social 1 trial: {s1_firstfive_time_calc}')

s1_last_ts = cam_ts.loc[cam_ts['Frame Number']==last_s1_frame, 'Time Stamp (ms)'].values[0]
s1_lastfive_time_calc = (s1_last_ts - s1_fivemin_ts)/1000

print(f'real time for estimated last five minutes (300 seconds) of social 1 trial: {s1_lastfive_time_calc}')

In [ ]:
s2_start_ts = cam_ts.loc[cam_ts['Frame Number']==first_s2_frame, 'Time Stamp (ms)'].values[0]
s2_firstfive_time_calc = (s2_fivemin_ts - s2_start_ts)/1000

print(f'real time for estimated first five minutes (300 seconds) of social 2 trial: {s2_firstfive_time_calc}')

s2_last_ts = cam_ts.loc[cam_ts['Frame Number']==last_s2_frame, 'Time Stamp (ms)'].values[0]
s2_lastfive_time_calc = (s2_last_ts - s2_fivemin_ts)/1000

print(f'real time for estimated last five minutes (300 seconds) of social 2 trial: {s2_lastfive_time_calc}')

In [ ]:
#create new raster dfs for each five minute chunk
s1_firstfive_df = rdf.iloc[:,first_s1_frame:s1_fivemin_frame].copy()
s1_lastfive_df = rdf.iloc[:,s1_fivemin_frame:last_s1_frame].copy()

s2_firstfive_df = rdf.iloc[:,first_s2_frame:s2_fivemin_frame].copy()
s2_lastfive_df = rdf.iloc[:,s2_fivemin_frame:last_s2_frame].copy()

In [ ]:
#compile into dict for easier looping in following analyses
five_min_dict = {
    's1_first_five_min': s1_firstfive_df,
    's1_last_five_min': s1_lastfive_df,
    's2_first_five_min': s2_firstfive_df,
    's2_last_five_min': s2_lastfive_df
}

number of active interactions w partner or novel

In [ ]:
#define function to get the partner and novel bouts for any df
def act_bouts(df,fr=cam_fr):
    pact_frames = df.loc[:,df.loc['partner_active']==1]
    pactcols = np.asarray(pact_frames.columns).astype(int)
    pact_bouts = utils.make_behavior_bouts(pactcols,fr=fr)

    nact_frames = df.loc[:,df.loc['novel_active']==1]
    nactcols = np.asarray(nact_frames.columns).astype(int)
    nact_bouts = utils.make_behavior_bouts(nactcols,fr=fr)

    return pact_bouts, pactcols, nact_bouts, nactcols

In [ ]:
#define percent pacts function
def num_acts(pacts,nacts):
    lpacts = len(pacts) #number of partner interactions
    lnacts = len(nacts) #number of novel interactions

    if lpacts + lnacts > 0: 
        percent_pacts = lpacts/(lpacts+lnacts) #percentage of interactions that are partner interactions
    else:
        percent_pacts = np.nan
        
    return lpacts, lnacts, percent_pacts

In [ ]:
#bouts for all active interactions
pacts, pactcols, nacts, nactcols = act_bouts(rdf,fr=cam_fr)

In [ ]:
lpacts, lnacts, percent_pacts = num_acts(pacts,nacts)
print(lpacts)
print(lnacts)
print(percent_pacts)

In [ ]:
#save
resdict['n_active_partner_ints']=lpacts
resdict['n_active_novel_ints']=lnacts
resdict['percent_active_ints_with_partner']=percent_pacts

In [ ]:
resdict

In [ ]:
pacts

Number of active interactions with partner or novel, each five minutes

In [ ]:
#modify this to add an empty dict to add each bouts df to for each interval
pacts_five_min = {}
nacts_five_min = {}
pact_cols_five_min = {}
nact_cols_five_min = {}


for interval_name, interval_df in five_min_dict.items():
    pacts_interval, pactcols_interval, nacts_interval, nactcols_interval = act_bouts(interval_df)
    pacts_five_min[f'{interval_name}_pacts'] = pacts_interval
    nacts_five_min[f'{interval_name}_nacts'] = nacts_interval
    pact_cols_five_min[f'{interval_name}_pact_cols'] = pactcols_interval
    nact_cols_five_min[f'{interval_name}_nact_cols'] = nactcols_interval
    
    lpacts_interval, lnacts_interval, percent_pacts_interval = num_acts(pacts_interval, nacts_interval)
    
    # Save with descriptive names
    resdict[f'{interval_name}_n_partner_ints'] = lpacts_interval
    resdict[f'{interval_name}_n_novel_ints'] = lnacts_interval
    resdict[f'{interval_name}_percent_partner_ints'] = percent_pacts_interval

In [ ]:
resdict

Average length of active interaction with partner or novel

In [ ]:
avg_lpact = pacts['length'].mean()
avg_lnact = nacts['length'].mean()

rat_avgacts = avg_lpact/avg_lnact

print(avg_lpact)
print(avg_lnact)
print(rat_avgacts)

In [ ]:
#save
resdict['avg_active_partner_int_length']=avg_lpact
resdict['avg_active_novel_int_length']=avg_lnact
resdict['ratio_avg_partner_to_novel_int_length']=rat_avgacts

In [ ]:
#now do the same for all five minute intervals:

for interval_name in five_min_dict.keys():
    # Get the stored bout dataframes for this interval
    pacts_interval = pacts_five_min[f'{interval_name}_pacts']
    nacts_interval = nacts_five_min[f'{interval_name}_nacts']
    
    # Calculate average partner interaction length
    if len(pacts_interval) > 0:
        avg_lpact_interval = pacts_interval['length'].mean()
        resdict[f'{interval_name}_avg_partner_int_length'] = avg_lpact_interval
    else:
        avg_lpact_interval = np.nan
        resdict[f'{interval_name}_avg_partner_int_length'] = np.nan
    
    # Calculate average novel interaction length
    if len(nacts_interval) > 0:
        avg_lnact_interval = nacts_interval['length'].mean()
        resdict[f'{interval_name}_avg_novel_int_length'] = avg_lnact_interval
    else:
        avg_lnact_interval = np.nan
        resdict[f'{interval_name}_avg_novel_int_length'] = np.nan
    
    # Calculate ratio of average lengths (only if both exist)
    if len(pacts_interval) > 0 and len(nacts_interval) > 0:
        rat_avgacts_interval = avg_lpact_interval / avg_lnact_interval
        resdict[f'{interval_name}_ratio_avg_partner_to_novel_length'] = rat_avgacts_interval
    else:
        resdict[f'{interval_name}_ratio_avg_partner_to_novel_length'] = np.nan

In [ ]:
resdict

Percentage of active interaction frames w partner or novel (=partner preference score / social recognition score)

In [ ]:
n_pframes = len(pactcols)
n_nframes = len(nactcols)

percent_intframes = n_pframes/(n_pframes+n_nframes)

print(n_pframes)
print(n_nframes)
print(percent_intframes)

In [ ]:
ppref_index = (n_pframes-n_nframes)/(n_pframes+n_nframes)
ppref_index

In [ ]:
#save
resdict['percent_partner_int_frames']=percent_intframes
resdict['partner_preference_index']=ppref_index

In [ ]:
#now same for each five minute interaction chunk

In [ ]:
for interval_name in five_min_dict.keys():
    num_pact_frames_interval = len(pact_cols_five_min[f'{interval_name}_pact_cols'])
    num_nact_frames_interval = len(nact_cols_five_min[f'{interval_name}_nact_cols'])

    if (num_pact_frames_interval + num_nact_frames_interval) > 0:
        percent_intframes_interval = num_pact_frames_interval/(num_pact_frames_interval+num_nact_frames_interval)
        ppref_index_interval = (num_pact_frames_interval-num_nact_frames_interval)/(num_pact_frames_interval+num_nact_frames_interval)
    else:
        percent_intframes_interval = np.nan
        ppref_index_interval = np.nan

    resdict[f'{interval_name}_percent_partner_int_frames'] = percent_intframes_interval
    resdict[f'{interval_name}_partner_preference_index']=ppref_index_interval

In [ ]:
resdict

Proportion of frames in either stimulus animal's chamber to active frames with that animal -- measure of passive vs. active interaction

In [ ]:
def prop_chamber_frames_active(df):
    #function to figure out what proprortion of frames in each chamber were spent actively interacting with the cup/stimulus under the cup
    pchamb_frames=df.loc[:,df.loc['partner_chamb']==1].copy()
    n_pcfs = pchamb_frames.shape[1]

    pact_frames = pchamb_frames.loc[:,pchamb_frames.loc['active']==1]
    n_pactfs = pact_frames.shape[1]

    if n_pcfs>0:
        prop_pchamb_act = n_pactfs/n_pcfs
    else:
        prop_pchamb_act = np.nan
    
    nchamb_frames=df.loc[:,df.loc['novel_chamb']==1].copy()
    n_ncfs = nchamb_frames.shape[1]

    nact_frames = nchamb_frames.loc[:,nchamb_frames.loc['active']==1]
    n_nactfs = nact_frames.shape[1]

    if n_ncfs>0:
        prop_nchamb_act = n_nactfs/n_ncfs
    else:
        prop_nchamb_act = np.nan

    return prop_pchamb_act, prop_nchamb_act

In [ ]:
prop_pchamb_act_total, prop_nchamb_act_total = prop_chamber_frames_active(rdf)
print(prop_pchamb_act_total)
print(prop_nchamb_act_total)
resdict['proportion_partner_chamber_frames_active']=prop_pchamb_act_total
resdict['proportion_novel_chamber_frames_active']=prop_nchamb_act_total

In [ ]:
#same for each five minute chunk

for interval_name, interval_df in five_min_dict.items():
    interval_prop_pchamb_act, interval_prop_nchamb_act = prop_chamber_frames_active(interval_df)
    resdict[f'{interval_name}_proportion_partner_chamber_frames_active']=interval_prop_pchamb_act
    resdict[f'{interval_name}_proportion_novel_chamber_frames_active']=interval_prop_nchamb_act

In [ ]:
resdict

Same metrics but now with chamber time instead of active interaction time

In [ ]:
def get_chamber_visit_metrics(df,fr=cam_fr):
    chamb_res_dict = {}
    
    #get data structures for partner chamber visits
    pchamb_frames=df.loc[:,df.loc['partner_chamb']==1].copy()
    pchambcols = np.asarray(pchamb_frames.columns).astype(int)

    #and novel chamber visits
    nchamb_frames=df.loc[:,df.loc['novel_chamb']==1].copy()
    nchambcols = np.asarray(nchamb_frames.columns).astype(int)

    if len(pchambcols) == 0 and len(nchambcols) == 0:
        #no chamber visits -> return null results
        chamb_res_dict['num_pchamb_visits'] = 0
        chamb_res_dict['num_nchamb_visits'] = 0
        chamb_res_dict['percent_pchamb_visits'] = np.nan
        chamb_res_dict['avg_partner_chamber_visit_length'] = np.nan
        chamb_res_dict['avg_novel_chamber_visit_length'] = np.nan
        chamb_res_dict['ratio_avg_partner_to_novel_chamber_visit_length'] = np.nan
        chamb_res_dict['percent_partner_chamber_frames'] = np.nan
        chamb_res_dict['partner_preference_index_chambertime'] = np.nan
        return chamb_res_dict

    #create novel and partner chamber bout dfs
    if len(pchambcols) > 0:
        pchamb_visits = utils.make_behavior_bouts(pchambcols,fr)
    else:
        pchamb_visits = pd.DataFrame(columns=['start','end','length'])
        
    if len(nchambcols) > 0:
        nchamb_visits = utils.make_behavior_bouts(nchambcols,fr)
    else:
        nchamb_visits = pd.DataFrame(columns=['start','end','length'])

    #get number of chamber visits and percentage of total chamber visits to the partner chamber
    lpchambs, lnchambs, percent_pchambs = num_acts(pchamb_visits,nchamb_visits)

    #save results
    chamb_res_dict['num_pchamb_visits']=lpchambs
    chamb_res_dict['num_nchamb_visits']=lnchambs
    chamb_res_dict['percent_pchamb_visits']=percent_pchambs

    #average length of visit to each chamber + ratio
    if len(pchamb_visits) > 0:
        avg_lpchamb = pchamb_visits['length'].mean()
    else:
        avg_lpchamb = np.nan
        
    if len(nchamb_visits) > 0:
        avg_lnchamb = nchamb_visits['length'].mean()
    else:
        avg_lnchamb = np.nan    

    if len(pchamb_visits) > 0 and len(nchamb_visits) > 0:
        rat_avgchambs = avg_lpchamb / avg_lnchamb
    else:
        rat_avgchambs = np.nan
        
    #save results
    chamb_res_dict['avg_partner_chamber_visit_length']=avg_lpchamb
    chamb_res_dict['avg_novel_chamber_visit_length']=avg_lnchamb
    chamb_res_dict['ratio_avg_partner_to_novel_chamber_visit_length']=rat_avgchambs

    #total # of frames in partner or novel chamber and calulate preference score and preference index
    num_pchambframes = len(pchambcols)
    num_nchambframes = len(nchambcols)
    total_chambframes = num_pchambframes + num_nchambframes

    if total_chambframes > 0:
        percent_chambframes = num_pchambframes / total_chambframes
        ppref_index_chamb = (num_pchambframes - num_nchambframes) / total_chambframes
    else:
        percent_chambframes = np.nan
        ppref_index_chamb = np.nan
     
    #save results
    chamb_res_dict['percent_partner_chamber_frames']=percent_chambframes
    chamb_res_dict['partner_preference_index_chambertime']=ppref_index_chamb
    
    return chamb_res_dict

In [ ]:
overall_chamb_metrics = get_chamber_visit_metrics(rdf,fr=cam_fr)

In [ ]:
#concatenate the chamber dict to results dictionary
for key, value in overall_chamb_metrics.items():
    resdict[key]=value

In [ ]:
#loop through each five minute chunk
for interval_name, interval_df in five_min_dict.items():
    interval_chamb_metrics = get_chamber_visit_metrics(interval_df, fr=cam_fr)
    
    # Add each metric to resdict with interval name prefix
    for key, value in interval_chamb_metrics.items():
        resdict[f'{interval_name}_{key}'] = value

In [ ]:
resdict

Latency to first active interaction w either animal

In [ ]:
pactive_arr = np.asarray(social1_df.loc['partner_active'])
plat = np.where(pactive_arr == 1)[0][0]
plat_sec = plat/cam_fr

nactive_arr = np.asarray(social1_df.loc['novel_active'])
nlat = np.where(nactive_arr == 1)[0][0]
nlat_sec = nlat/cam_fr

print(plat_sec)
print(nlat_sec)

In [ ]:
#save
resdict['latency_to_first_partner_int']=plat_sec
resdict['latency_to_first_novel_int']=nlat_sec

In [ ]:
resdict

Shortest inter-bout-interval between a novel and a partner bout

In [ ]:
#make new partner and novel bout dfs with 1) zero-indexing (as rdf is 1-indexed as it originally came from matlab) and
#2) add labels for which trial each bout came from
def reshape_bouts_df(bouts_df, first_s1_frame, last_s1_frame, 
                     first_s2_frame, last_s2_frame):
    #zero-index
    bouts_df_zi = bouts_df.copy()
    bouts_df_zi['start']=bouts_df_zi['start']-1
    bouts_df_zi['end']=bouts_df_zi['end']-1

    #add trial label
    bouts_df_zi['trial']='unknown'
    bouts_df_zi.loc[(bouts_df_zi['start'] >= first_s1_frame) & (bouts_df_zi['start'] <= last_s1_frame), 'trial'] = 's1'
    bouts_df_zi.loc[(bouts_df_zi['start'] >= first_s2_frame) & (bouts_df_zi['start'] <= last_s2_frame), 'trial'] = 's2'

    return bouts_df_zi

In [ ]:
pacts_zi = reshape_bouts_df(pacts, first_s1_frame, last_s1_frame, 
                     first_s2_frame, last_s2_frame)
nacts_zi = reshape_bouts_df(nacts, first_s1_frame, last_s1_frame, 
                     first_s2_frame, last_s2_frame)

In [ ]:
def get_interbout_intervals(pacts, nacts, fr=cam_fr):
    """
    Get intervals between temporally neighboring partner and novel interaction bouts
    
    Only counts gaps where a partner bout is immediately followed by a novel bout
    (or vice versa) with no other bouts in between, AND both bouts are in the same trial.
    
    Parameters:
    -----------
    pacts : pd.DataFrame
        Partner active bouts with 'start', 'end', 'length', 'trial' columns
    nacts : pd.DataFrame
        Novel active bouts with 'start', 'end', 'length', 'trial' columns
        
    Returns:
    --------
    intervals : np.array
        Array of gap durations between neighboring partner<->novel switches
        (excludes intervals spanning across trials)
    """
    
    if len(pacts) == 0 or len(nacts) == 0:
        return np.array([])
    
    # Create combined dataframe with bout identity labels
    pacts_labeled = pacts.copy()
    pacts_labeled['identity'] = 'partner'
    
    nacts_labeled = nacts.copy()
    nacts_labeled['identity'] = 'novel'
    
    # Combine and sort by start time
    all_bouts = pd.concat([pacts_labeled, nacts_labeled], ignore_index=True)
    all_bouts = all_bouts.sort_values('start').reset_index(drop=True)
    
    # Find intervals between consecutive bouts of different identities
    intervals = []
    
    for i in range(len(all_bouts) - 1):
        current_bout = all_bouts.iloc[i]
        next_bout = all_bouts.iloc[i + 1]
        
        # Only record interval if:
        # 1. Switching between partner and novel (different identity)
        # 2. Both bouts are in the SAME trial (skip cross-trial gaps)
        if (current_bout['identity'] != next_bout['identity'] and 
            current_bout['trial'] == next_bout['trial']):
            gap = next_bout['start'] - current_bout['end']
            gap = gap/fr
            intervals.append(gap)
    
    return np.array(intervals)

In [ ]:
intervals = get_interbout_intervals(pacts_zi, nacts_zi, cam_fr)
intervals

In [ ]:
if len(intervals) > 0:
    resdict['min_int_type_switch_interval'] = intervals.min()
    resdict['mean_int_type_switch_interval'] = intervals.mean()
    resdict['median_int_type_switch_interval'] = np.median(intervals)
    resdict['num_int_type_switches'] = len(intervals)
else:
    resdict['min_int_type_switch_interval'] = np.nan
    resdict['mean_int_type_switch_interval'] = np.nan
    resdict['median_int_type_switch_interval'] = np.nan
    resdict['num_int_type_switches'] = 0

In [ ]:
resdict

Save out results dictionary

In [ ]:
respath = os.path.join(behpath, 'behavior_results.pkl')

with open(respath, 'wb') as f:
    pickle.dump(resdict,f)